# chessRL — training on Colab

Trains the self-play agent in **resumable sessions**. Everything (config, network, optimiser,
replay buffer, metrics, snapshots) lives in a run folder on Google Drive, so you can stop the
runtime at any time and continue later by re-running the last cell.

Runtime → Change runtime type → **GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/chessRL/runs/colab-main'   # change the name for a new run

In [ ]:
!git clone https://github.com/fragompul/chessRL.git /content/chessRL 2>/dev/null || git -C /content/chessRL pull
%cd /content/chessRL
!pip install -q -e ".[rl]"
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Train one session
`--iterations` is *per session*. The first call creates the run from the `colab` preset;
later calls resume from `latest.pt`. Use `--max-minutes` to stay inside Colab's session limits.

In [ ]:
!python -m chessrl train --run $RUN_DIR --preset colab --iterations 20 --max-minutes 150

## Progress

In [ ]:
!python -m chessrl status --run $RUN_DIR

In [ ]:
import json, pathlib
rows = [json.loads(l) for l in pathlib.Path(RUN_DIR, 'metrics.jsonl').read_text().splitlines()]
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r['iteration'] for r in rows if 'loss' in r], [r['loss'] for r in rows if 'loss' in r]); ax[0].set_title('loss')
ev = [(r['iteration'], r['eval_score']) for r in rows if 'eval_score' in r]
if ev: ax[1].plot(*zip(*ev), marker='o'); ax[1].set_title('score vs alpha-beta baseline')
plt.show()

## Next steps
* Snapshots (`ckpt/iter_*.pt`) are weights-only files: download them to your machine and use them with
  `python -m chessrl play --opponent ckpt:path/to/iter_000100.pt`.
* Calibrate Elo: `python -m chessrl elo --run $RUN_DIR` (see docs/EVALUATION.md).